# Hotel Booking Cancellation Analysis

Projeto educacional de análise exploratória e Machine Learning para investigar padrões associados ao cancelamento de reservas hoteleiras e construir um modelo de classificação.

Dataset: **Hotel Booking Demand** (`jessemostipak/hotel-booking-demand`).


## 1. Preparação do ambiente e carregamento

O dataset é baixado com KaggleHub e armazenado em `data/raw/`, mantendo os dados brutos fora do versionamento.


In [ ]:
from pathlib import Path
import shutil
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
)

RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_FILE = RAW_DATA_DIR / "hotel_bookings.csv"

if not RAW_DATA_FILE.exists():
    downloaded_path = Path(
        kagglehub.dataset_download("jessemostipak/hotel-booking-demand")
    )
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(downloaded_path / "hotel_bookings.csv", RAW_DATA_FILE)

df_origin = pd.read_csv(RAW_DATA_FILE)
df = df_origin.copy()
df.head()


## 2. Tratamento dos dados

A preparação inclui tratamento de valores ausentes, criação de indicadores para `agent` e `company` e remoção de registros duplicados.


In [ ]:
df["children"] = df["children"].fillna(0).astype(int)
df["country"] = df["country"].fillna("UNK")

df["has_agent"] = df["agent"].notnull().astype(int)
df["has_company"] = df["company"].notnull().astype(int)

df = df.drop_duplicates()

df.isnull().sum()[df.isnull().sum() > 0]


## 3. Análise exploratória

A investigação avalia distribuição de cancelamentos, antecedência da reserva (`lead_time`), diária média (`adr`), sazonalidade e outras características operacionais.


In [ ]:
cancel_distribution = pd.DataFrame({
    "total_reservations": df["is_canceled"].value_counts(),
    "percentage": (df["is_canceled"].value_counts(normalize=True) * 100).round(2),
})

lead_time_by_status = df.groupby("is_canceled")["lead_time"].median()
adr_by_status = df.groupby("is_canceled")["adr"].median()

lead_time_bins = [0, 30, 90, 180, df["lead_time"].max()]
lead_time_labels = ["0-30 dias", "31-90 dias", "91-180 dias", "181+ dias"]
df["lead_time_range"] = pd.cut(
    df["lead_time"], bins=lead_time_bins, labels=lead_time_labels
)
lead_time_cancel_rates = (
    df.groupby("lead_time_range", observed=False)["is_canceled"]
      .mean().mul(100).round(2)
)

cancel_distribution, lead_time_by_status, adr_by_status, lead_time_cancel_rates


## 4. Preparação para modelagem

As colunas `reservation_status` e `reservation_status_date` são removidas por risco de **data leakage**. `agent` e `company` também são removidas após a criação dos indicadores de presença.

A variável `country` tem a cardinalidade reduzida para os dez países mais frequentes; `adr` é limitado ao percentil 99; e variáveis categóricas são codificadas com one-hot encoding.


In [ ]:
columns_to_drop = [
    "reservation_status",
    "reservation_status_date",
    "agent",
    "company",
]
df = df.drop(columns=columns_to_drop)

y = df["is_canceled"]
X = df.drop(columns="is_canceled")

top_countries = X["country"].value_counts().head(10).index
X["country"] = X["country"].apply(
    lambda x: x if x in top_countries else "OTH"
)

adr_upper_limit = df["adr"].quantile(0.99)
X["adr"] = X["adr"].clip(lower=0, upper=adr_upper_limit)
X["has_waiting_list"] = (X["days_in_waiting_list"] > 0).astype(int)

X = pd.get_dummies(X, drop_first=True).astype(int)
print(f"Registros: {X.shape[0]} | Features: {X.shape[1]}")


## 5. Treino e avaliação

O conjunto é dividido de forma estratificada entre treino e teste. O modelo utilizado é um `RandomForestClassifier`.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1986,
    stratify=y,
)

rf_model = RandomForestClassifier(random_state=1986)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(classification_report(y_test, y_pred))


### Resultado registrado na execução original

- **Accuracy:** 84,47%
- **Precision:** 75,85%
- **Recall:** 63,81%
- **F1-Score:** 69,31%

Matriz de confusão:

```text
[[11699,  976],
 [ 1739, 3066]]
```

O modelo teve melhor desempenho na identificação de reservas não canceladas. O recall da classe de cancelamento indica espaço para melhoria na captura de cancelamentos reais.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de Confusão")
plt.xlabel("Predição do Modelo")
plt.ylabel("Valor Real")
plt.show()

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_,
}).sort_values(by="importance", ascending=False)

top_features = feature_importance.head(10)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_features, x="importance", y="feature")
plt.title("Top 10 Features Mais Importantes")
plt.xlabel("Importância")
plt.ylabel("Feature")
plt.show()

top_features


## 6. Conclusão

Os sinais mais relevantes incluem `lead_time`, `adr`, características temporais da chegada, solicitações especiais e variáveis operacionais da reserva.

O projeto demonstra um fluxo completo de análise: entendimento dos dados, tratamento, EDA, prevenção de leakage, feature engineering, codificação, divisão estratificada, treinamento de Random Forest e avaliação por múltiplas métricas.
